In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append("../src")

from derivatives import EuropeanCall, EuropeanPut, EuropeanCallGreeks, AmericanPut, AsianCall
from portfolio import EquityPosition, Portfolio
from yieldcurve import YieldCurve

In [8]:
# ── Load BHP Data ─────────────────────────────────────
bhp = pd.read_csv("../data/Market_Data_BHP_prices.csv", parse_dates=["Date"])
bhp = bhp.sort_values("Date").reset_index(drop=True)

# Compute daily returns
bhp["Return"] = bhp["Close"].pct_change()
bhp_returns   = bhp["Return"].dropna()

# Key parameters from the data
S0    = bhp["Close"].iloc[-1]          # Latest closing price (57.33)
sigma = bhp_returns.std() * np.sqrt(252)  # Annualised historical volatility
K     = round(S0)                      # ATM strike, rounded to nearest dollar
T     = 1.0                            # 1-year expiry

print(f"S0    = ${S0:.2f}")
print(f"K     = ${K:.2f}")
print(f"T     = {T} year")
print(f"sigma = {sigma:.4f} ({sigma*100:.2f}%)")

S0    = $57.33
K     = $57.00
T     = 1.0 year
sigma = 0.2492 (24.92%)


In [9]:
# ── 1. Create the yield curve object ──────────────────
# Australian government zero rates (approximate, May 2026)
maturities = [0.25, 0.5, 1.0, 2.0, 5.0]
zero_rates  = [0.041, 0.042, 0.043, 0.044, 0.045]

curve = YieldCurve(maturities=maturities, zero_rates=zero_rates)
curve

In [10]:
equity = EquityPosition(ticker="BHP", spot=S0)

# Ensure call/put are instantiated (use existing curve if present)
if 'curve' not in globals():
    maturities = [0.25, 0.5, 1.0, 2.0, 5.0]
    zero_rates  = [0.041, 0.042, 0.043, 0.044, 0.045]
    curve = YieldCurve(maturities=maturities, zero_rates=zero_rates)

call = EuropeanCall(S0=S0, K=K, T=T, sigma=sigma, yield_curve=curve)
put = EuropeanPut(S0=S0, K=K, T=T, sigma=sigma, yield_curve=curve)

print(f"Equity price: {round(equity.price(), 4)}")
print(f"Equity delta: {round(equity.delta(), 4)}")
print()
print(f"Call price:   {round(call.price(), 4)}")
print(f"Call delta:   {round(call.delta(), 4)}")
print()
print(f"Put price:    {round(put.price(), 4)}")
print(f"Put delta:    {round(put.delta(), 4)}")

Equity price: 57.33
Equity delta: 1.0

Call price:   7.0201
Call delta:   0.6256

Put price:    4.2911
Put delta:    -0.3744
